# Diabetes Classification

Compare a baseline XGBoost classifier with a class-weighted model tuned for recall. Evaluate both on the same held-out rows and inspect the tuned model's probability estimates.

## 1. Load the data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibrationDisplay
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    auc,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

In [ ]:
data_path = Path("data/diabetes.csv")
if not data_path.is_file():
    raise FileNotFoundError(f"Place the project CSV at {data_path} before running the notebook.")

diabetes_data = pd.read_csv(data_path)

In [ ]:
# Take a look at the first few rows of the data
diabetes_data.head()

In [ ]:
diabetes_data.describe()

Zero represents a missing measurement in five columns. Keep zeros in other columns, including `Pregnancies` and the target `Outcome`.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
diabetes_data[zero_as_missing] = diabetes_data[zero_as_missing].replace(0, np.nan)

## 2. Check missing values

In [ ]:
print(diabetes_data.isnull().sum())

Fit the median imputer inside each model pipeline so that it learns from the training rows available to that fit.

## 3. Split the data

In [ ]:
features = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
]
X = diabetes_data[features]
y = diabetes_data["Outcome"]

In [ ]:
# The original experiment used 80 training rows.
random_seed = 42
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, train_size=80, stratify=y, random_state=random_seed
)
print(f"Training rows: {len(X_train)}, validation rows: {len(X_valid)}")

## 4. Fit and evaluate the baseline

In [ ]:
baseline = make_pipeline(
    SimpleImputer(strategy="median"),
    XGBClassifier(random_state=random_seed, n_jobs=1),
)
baseline.fit(X_train, y_train)

In [ ]:
def validation_metrics(model):
    predictions = model.predict(X_valid)
    tn, fp, fn, tp = confusion_matrix(y_valid, predictions, labels=[0, 1]).ravel()

    def rate(numerator, denominator):
        return numerator / denominator if denominator else np.nan

    return {
        "Accuracy": accuracy_score(y_valid, predictions),
        "Sensitivity (recall)": rate(tp, tp + fn),
        "Specificity": rate(tn, tn + fp),
        "Precision (PPV)": rate(tp, tp + fp),
        "Negative predictive value": rate(tn, tn + fn),
    }

baseline_metrics = validation_metrics(baseline)
pd.Series(baseline_metrics, name="Baseline").round(4)

## 5. Tune the model for sensitivity

In [ ]:
# Use the training class ratio to weight the positive class.
class_counts = y_train.value_counts()
scale_pos_weight = class_counts.loc[0] / class_counts.loc[1]

tuning_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    XGBClassifier(
        random_state=random_seed,
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        n_jobs=1,
    ),
)

param_grid = {
    "xgbclassifier__max_depth": [3, 4, 5],
    "xgbclassifier__learning_rate": [0.05, 0.1],
    "xgbclassifier__n_estimators": [100, 200],
    "xgbclassifier__subsample": [0.7, 0.9],
    "xgbclassifier__colsample_bytree": [0.8, 1.0],
    "xgbclassifier__reg_lambda": [0.5, 1.0, 2.0],
}

grid_search = GridSearchCV(
    tuning_pipeline,
    param_grid,
    cv=5,
    scoring="recall",
    verbose=1,
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print(f"Best mean cross-validation sensitivity: {grid_search.best_score_:.4f}")

Grid search refits the selected pipeline on the full training set. Evaluate that pipeline on the same validation rows used for the baseline.

In [ ]:
tuned = grid_search.best_estimator_
pd.DataFrame(
    {"Baseline": baseline_metrics, "Tuned": validation_metrics(tuned)}
).round(4)

## 6. Evaluate the tuned model's probabilities

In [ ]:
y_pred_proba = tuned.predict_proba(X_valid)[:, 1]

roc_auc = roc_auc_score(y_valid, y_pred_proba)
precision, recall, _ = precision_recall_curve(y_valid, y_pred_proba)
pr_auc = auc(recall, precision)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC (trapezoidal): {pr_auc:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_valid, y_pred_proba)
ax1.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}', linewidth=2)
ax1.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax1.set_xlabel('False Positive Rate')
ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve')
ax1.legend()
ax1.grid(alpha=0.3)

# PR Curve
ax2.plot(recall, precision, label=f'PR-AUC = {pr_auc:.4f}', linewidth=2)
ax2.set_xlabel('Recall (Sensitivity)')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve')
ax2.legend()
ax2.grid(alpha=0.3)
ax2.set_xlim([0, 1])
ax2.set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

CalibrationDisplay.from_estimator(
    tuned,
    X_valid,
    y_valid,
    n_bins=10,
    ax=ax,
    strategy="uniform",
)

ax.set_title("Calibration Plot")
plt.tight_layout()
plt.show()

## Reflection and next experiment

This first generation establishes the 80-row XGBoost baseline. The historical tuning attempt showed that recall could fall even when accuracy held steady. The next generation compares training sizes and cutoffs on a shared holdout to see which changes affect ranking and errors.